<a href="https://colab.research.google.com/github/sudhans18/ABDA/blob/main/02_LaBSE_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U sentence-transformers pandas==2.2.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 14.5 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [36]:
PROJECT_DIR = "/content/drive/MyDrive/cmu_mosei"

READY_PATH = os.path.join(
    PROJECT_DIR,
    "CMU_MOSEI_LaBSE_ready_v2.pkl"
)

EMBEDDING_PATH = os.path.join(
    PROJECT_DIR,
    "CMU_MOSEI_LaBSE_embeddings.pkl"
)

print("Ready file:", READY_PATH)
print("Embedding output:", EMBEDDING_PATH)

Ready file: /content/drive/MyDrive/cmu_mosei/CMU_MOSEI_LaBSE_ready_v2.pkl
Embedding output: /content/drive/MyDrive/cmu_mosei/CMU_MOSEI_LaBSE_embeddings.pkl


In [37]:
ready_df = pd.read_pickle(READY_PATH)

print("Shape:", ready_df.shape)
print("\nColumns:")
print(ready_df.columns.tolist())

Shape: (23253, 13)

Columns:
['segment_id', 'label_idx', 'start', 'end', 'text', 'token_count', 'sentiment', 'happiness', 'sadness', 'anger', 'surprise', 'disgust', 'fear']


In [38]:
assert len(ready_df) == 23253
assert "text" in ready_df.columns
assert "token_count" in ready_df.columns

print("Number of rows:", len(ready_df))
print("Missing text:", ready_df["text"].isna().sum())
print("Empty text:", (ready_df["text"].str.strip() == "").sum())
print("Duplicate rows:", ready_df.duplicated().sum())

Number of rows: 23253
Missing text: 0
Empty text: 0
Duplicate rows: 0


In [39]:
for i in range(5):
    print(f"\nSample {i}")
    print(ready_df.iloc[i]["text"])


Sample 0
writer i see that a writer is somebody who has an incredible command of mechanics of the english language

Sample 1
key is part of the people that we use to solve those issues whether it's stretch or outdoor resistance or abrasions or different technical aspects that we really need to solve to get into new markets they've been able to bring solutions

Sample 2
businesses that we do they've been able to find solutions or at least bring some answers to the table

Sample 3
operations key polymer brings a technical aspect to our operation that we don't have internally we're

Sample 4
internally we're a huge user of adhesives for our operation called flocking and we don't have the technical inside compounding or the technical expertise to do these types of things key brings


In [41]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "sentence-transformers/LaBSE",
    device=device
)

print("Device:", device)
print("Model loaded successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Device: cuda
Model loaded successfully


In [42]:
embedding_dim = model.get_sentence_embedding_dimension()
max_length = model.max_seq_length

print("Embedding dimension:", embedding_dim)
print("Model max sequence length:", max_length)

Embedding dimension: 768
Model max sequence length: 256


/tmp/ipykernel_3733/608084918.py:1: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = model.get_sentence_embedding_dimension()


In [43]:
tokenizer = model.tokenizer

print("Tokenizer model max length:", tokenizer.model_max_length)

Tokenizer model max length: 256


In [45]:
print("Maximum token count:", ready_df["token_count"].max())
print("Samples > 256:", (ready_df["token_count"] > max_length).sum())

Maximum token count: 223
Samples > 256: 0


In [46]:
print(ready_df["token_count"].describe())

count    23253.000000
mean        25.477573
std         13.778297
min          3.000000
25%         16.000000
50%         22.000000
75%         31.000000
max        223.000000
Name: token_count, dtype: float64


Pilot Embedding (before encoding all 23k samples)

In [47]:
pilot_texts = ready_df["text"].iloc[:16].tolist()

pilot_embeddings = model.encode(
    pilot_texts,
    batch_size=16,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Pilot embedding shape:", pilot_embeddings.shape)

Pilot embedding shape: (16, 768)


In [48]:
print("Contains NaN:", np.isnan(pilot_embeddings).any())
print("Contains Inf:", np.isinf(pilot_embeddings).any())

norms = np.linalg.norm(pilot_embeddings, axis=1)

print("Minimum norm:", norms.min())
print("Maximum norm:", norms.max())

Contains NaN: False
Contains Inf: False
Minimum norm: 0.99999994
Maximum norm: 1.0


In [49]:
print("Embedding dimension:", len(pilot_embeddings[0]))
print("\nFirst 20 values:")
print(pilot_embeddings[0][:20])

Embedding dimension: 768

First 20 values:
[-0.01295004  0.004679   -0.0462564  -0.07088806 -0.00265373  0.01616439
 -0.0473111  -0.02446704 -0.04286124  0.01269871 -0.01097088  0.02197312
 -0.01705803 -0.06662656 -0.06371916 -0.04930012  0.02148882  0.01571123
 -0.00682016 -0.06145303]


Full Embedding Generation

In [50]:
embedding_df = ready_df.copy()

all_texts = embedding_df["text"].tolist()

print("Number of texts:", len(all_texts))

Number of texts: 23253


In [51]:
embeddings = model.encode(
    all_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding matrix shape:", embeddings.shape)

Batches:   0%|          | 0/727 [00:00<?, ?it/s]

Embedding matrix shape: (23253, 768)


Validating the full embedding matrix

In [52]:
assert embeddings.shape == (len(embedding_df), 768)

print("Shape validation passed.")
print("Shape:", embeddings.shape)

Shape validation passed.
Shape: (23253, 768)


In [53]:
print("NaN present:", np.isnan(embeddings).any())
print("Inf present:", np.isinf(embeddings).any())

assert not np.isnan(embeddings).any()
assert not np.isinf(embeddings).any()

print("Numerical validation passed.")

NaN present: False
Inf present: False
Numerical validation passed.


In [54]:
embedding_norms = np.linalg.norm(embeddings, axis=1)

print("Minimum norm:", embedding_norms.min())
print("Maximum norm:", embedding_norms.max())
print("Mean norm:", embedding_norms.mean())

Minimum norm: 0.9999999
Maximum norm: 1.0000001
Mean norm: 1.0


In [55]:
embedding_df["labse_embedding"] = list(embeddings)

print(embedding_df.shape)
print(embedding_df.columns.tolist())

(23253, 14)
['segment_id', 'label_idx', 'start', 'end', 'text', 'token_count', 'sentiment', 'happiness', 'sadness', 'anger', 'surprise', 'disgust', 'fear', 'labse_embedding']


In [56]:
embedding_lengths = embedding_df["labse_embedding"].apply(len)

print(embedding_lengths.value_counts())

assert (embedding_lengths == 768).all()

print("All embeddings are 768-dimensional.")

labse_embedding
768    23253
Name: count, dtype: int64
All embeddings are 768-dimensional.


In [57]:
embedding_df.to_pickle(EMBEDDING_PATH)

print("Saved successfully:")
print(EMBEDDING_PATH)

Saved successfully:
/content/drive/MyDrive/cmu_mosei/CMU_MOSEI_LaBSE_embeddings.pkl


In [62]:
import json

metadata = {
    "dataset": "CMU-MOSEI",
    "feature_type": "Semantic text embeddings",
    "model": "sentence-transformers/LaBSE",
    "embedding_dimension": int(embedding_dim),
    "model_max_sequence_length": int(max_length),
    "number_of_samples": int(len(embedding_df)),
    "excluded_samples_due_to_sequence_length": 6,
    "normalization": True,
    "device": device,
    "source_file": os.path.basename(READY_PATH),
    "output_file": os.path.basename(EMBEDDING_PATH)
}

METADATA_PATH = os.path.join(
    PROJECT_DIR,
    "CMU_MOSEI_LaBSE_embeddings_metadata.json"
)

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=4)

print("Metadata saved:")
print(METADATA_PATH)

Metadata saved:
/content/drive/MyDrive/cmu_mosei/CMU_MOSEI_LaBSE_embeddings_metadata.json


In [63]:
print("=" * 60)
print("CMU-MOSEI LaBSE EMBEDDING PIPELINE COMPLETE")
print("=" * 60)

print("Samples:", len(embedding_df))
print("Embedding dimension:", embedding_dim)
print("Max sequence length:", max_length)
print("Excluded > max length:", 6)
print("NaN:", np.isnan(embeddings).any())
print("Inf:", np.isinf(embeddings).any())
print("Normalized:", True)

print("\nOutput:")
print(EMBEDDING_PATH)

print("\nMetadata:")
print(METADATA_PATH)

CMU-MOSEI LaBSE EMBEDDING PIPELINE COMPLETE
Samples: 23253
Embedding dimension: 768
Max sequence length: 256
Excluded > max length: 6
NaN: False
Inf: False
Normalized: True

Output:
/content/drive/MyDrive/cmu_mosei/CMU_MOSEI_LaBSE_embeddings.pkl

Metadata:
/content/drive/MyDrive/cmu_mosei/CMU_MOSEI_LaBSE_embeddings_metadata.json
